In [1]:
import pandas as pd
import numpy as np
import time

print("Creating 1 million rows of city sensor data... please wait.")

# Generate mock data for a smart city
np.random.seed(42)
num_rows = 1_000_000

data = {
    'Timestamp': pd.date_range(start='2026-01-01', periods=num_rows, freq='s'),
    'Zone_ID': np.random.randint(1, 51, size=num_rows),
    'Temperature_F': np.random.uniform(60, 115, size=num_rows),
    'Air_Quality_PM25': np.random.uniform(5, 160, size=num_rows),
    'Citizen_Alerts': np.random.choice(['None', 'Heat complaint', 'Smog observed', 'Traffic delay'], size=num_rows, p=[0.90, 0.04, 0.03, 0.03])
}

df_mock = pd.DataFrame(data)
df_mock.to_csv('city_sensor_data.csv', index=False)
print("Success! Your file 'city_sensor_data.csv' is created and ready.")


Creating 1 million rows of city sensor data... please wait.
Success! Your file 'city_sensor_data.csv' is created and ready.


In [2]:
# --- TEST 1: STANDARD CPU PROCESSING ---
import pandas as pd
start_time = time.time()

# Load the data on CPU
df_cpu = pd.read_csv('city_sensor_data.csv')

# Calculate an Emergency Risk Score on CPU
df_cpu['Risk_Score'] = (df_cpu['Temperature_F'] * 0.6) + (df_cpu['Air_Quality_PM25'] * 0.4)

# Find dangerous zones
critical_zones_cpu = df_cpu[df_cpu['Risk_Score'] > 85].groupby('Zone_ID').mean(numeric_only=True)

cpu_duration = time.time() - start_time
print(f"CPU Processing Time: {cpu_duration:.4f} seconds")


CPU Processing Time: 1.3472 seconds


In [3]:
# --- TEST 2: NVIDIA GPU ACCELERATION ---
# This single line activates the NVIDIA GPU for all your standard Pandas code!
%load_ext cudf.pandas
import pandas as pd

start_time = time.time()

# Load the data on GPU
df_gpu = pd.read_csv('city_sensor_data.csv')

# Calculate the exact same Risk Score on GPU
df_gpu['Risk_Score'] = (df_gpu['Temperature_F'] * 0.6) + (df_gpu['Air_Quality_PM25'] * 0.4)

# Find dangerous zones on GPU
critical_zones_gpu = df_gpu[df_gpu['Risk_Score'] > 85].groupby('Zone_ID').mean(numeric_only=True)

gpu_duration = time.time() - start_time
print(f"NVIDIA GPU Processing Time: {gpu_duration:.4f} seconds")


NVIDIA GPU Processing Time: 0.7447 seconds


In [4]:
!pip install -q google-genai


In [5]:
# Grab the top 3 highest risk city zones calculated by your NVIDIA GPU pipeline
top_critical_zones = critical_zones_gpu.sort_values(by='Risk_Score', ascending=False).head(3)
zone_summary_text = top_critical_zones[['Temperature_F', 'Air_Quality_PM25', 'Risk_Score']].to_string()

print("Sending accelerated data to Gemini AI Assistant...\n")

# This mimics the exact response structure your Gemini 1.5 Pro model generates for the dashboard
prompt = f"""
Act as an AI Decision Intelligence Platform for City Stakeholders.
Based on this accelerated real-time risk data calculated on NVIDIA GPUs:
{zone_summary_text}

Generate an Emergency Dispatch Recommendation.
"""

simulated_gemini_response = f"""
======================================================================
🤖 AI DECISION INTELLIGENCE PLATFORM - EMERGENCY DISPATCH DIRECTIVE
======================================================================
[STATUS: CRITICAL ANOMALIES DETECTED VIA NVIDIA ACCELERATED PIPELINE]

Based on real-time data processing, the following urgent actions are recommended:

1. 📍 DEPLOY COOLING CENTERS (Zone {top_critical_zones.index[0]}):
   - Extreme Heat & Air Pollution combined index has breached safe thresholds.
   - Action: Open public air-conditioned facilities and distribute water.

2. ⚠️ AIR QUALITY ALERT (Zone {top_critical_zones.index[1]} & Zone {top_critical_zones.index[2]}):
   - High PM2.5 particulate levels detected.
   - Action: Issue public health warnings advising sensitive groups to stay indoors.

3. 🔄 AUTOMATED WORKFLOW TRIGGERED:
   - Notification packets containing this risk assessment have been dispatched to
     Emergency Response Teams and City Stakeholders.
======================================================================
"""

print(simulated_gemini_response)


Sending accelerated data to Gemini AI Assistant...


🤖 AI DECISION INTELLIGENCE PLATFORM - EMERGENCY DISPATCH DIRECTIVE
[STATUS: CRITICAL ANOMALIES DETECTED VIA NVIDIA ACCELERATED PIPELINE]

Based on real-time data processing, the following urgent actions are recommended:

1. 📍 DEPLOY COOLING CENTERS (Zone 23):
   - Extreme Heat & Air Pollution combined index has breached safe thresholds.
   - Action: Open public air-conditioned facilities and distribute water.

2. ⚠️ AIR QUALITY ALERT (Zone 39 & Zone 8):
   - High PM2.5 particulate levels detected.
   - Action: Issue public health warnings advising sensitive groups to stay indoors.

3. 🔄 AUTOMATED WORKFLOW TRIGGERED:
   - Notification packets containing this risk assessment have been dispatched to 
     Emergency Response Teams and City Stakeholders.

